# Confidence lab · exploring Mind2Web

Runs in the lab's own environment (`labs/.venv`): `./labs/lab.sh open` creates it and opens this notebook in Jupyter Lab.

`data/mind2web/train_0.json`: the first ~100 train tasks of [Mind2Web](https://huggingface.co/datasets/osunlp/Mind2Web).

- **task** = a goal on a website ("Find a flight from NYC to Paris")
- **step** = one action of that task: an operation (CLICK / TYPE / SELECT, + a value) on one element of the page

## Load

The first run reads the 616 MB JSON once and saves two light tables (no HTML). Then it's instant.

In [8]:
import json
import time
from pathlib import Path

import pandas as pd

DATA = Path("..") / "data" / "mind2web"
RAW = DATA / "train_0.json"            # downloaded by hand (616 MB, ~100 tasks, with every page's HTML)
TASKS = DATA / "tasks.parquet"         # made once from RAW: one row per task, no HTML
STEPS = DATA / "steps.parquet"         # made once from RAW: one row per step, no HTML


def target_of(action: dict) -> dict:
    """The step's right element: the one the annotator picked, else the first positive."""
    positives = action["pos_candidates"]
    if not positives:
        return {}
    picked = [candidate for candidate in positives if candidate["is_original_target"]]
    return (picked or positives)[0]


def make_tables() -> None:
    """Read the big JSON once (Python's C parser: seconds) and keep everything but the HTML."""
    start = time.monotonic()
    with open(RAW) as file:
        raw_tasks = json.load(file)
    print(f"read {len(raw_tasks)} tasks in {time.monotonic() - start:.1f}s")

    tasks, steps = [], []
    for task in raw_tasks:
        tasks.append({
            "annotation_id": task["annotation_id"],
            "website": task["website"],
            "domain": task["domain"],
            "subdomain": task["subdomain"],
            "task": task["confirmed_task"],
            "steps": len(task["actions"]),
        })
        for number, (action, readable) in enumerate(zip(task["actions"], task["action_reprs"]), start=1):
            target = target_of(action)
            steps.append({
                "annotation_id": task["annotation_id"],
                "action_uid": action["action_uid"],
                "website": task["website"],
                "step": number,
                "action": readable,
                "op": action["operation"]["op"],
                "original_op": action["operation"]["original_op"],
                "value": action["operation"]["value"],
                "target_tag": target.get("tag"),
                "target_attributes": target.get("attributes"),  # JSON text
                "target_node_id": target.get("backend_node_id"),
                "right_elements": len(action["pos_candidates"]),
                "other_elements": len(action["neg_candidates"]),
                "html_chars": len(action["cleaned_html"]),
            })

    pd.DataFrame(tasks).to_parquet(TASKS)
    pd.DataFrame(steps).to_parquet(STEPS)
    print(f"saved {len(tasks)} tasks and {len(steps)} steps in {time.monotonic() - start:.1f}s")


if not (TASKS.exists() and STEPS.exists()):
    make_tables()

tasks = pd.read_parquet(TASKS)
steps = pd.read_parquet(STEPS)
print(len(tasks), "tasks ·", len(steps), "steps")

100 tasks · 774 steps


## Overview

In [9]:
pd.Series({
    "tasks": len(tasks),
    "steps": len(steps),
    "websites": tasks["website"].nunique(),
    "domains": tasks["domain"].nunique(),
    "subdomains": tasks["subdomain"].nunique(),
    "steps per task (mean)": round(steps.groupby("annotation_id").size().mean(), 1),
})

tasks                    100.0
steps                    774.0
websites                  38.0
domains                    3.0
subdomains                17.0
steps per task (mean)      7.7
dtype: float64

In [10]:
tasks.head(5)

,annotation_id,website,domain,subdomain,task,steps
0,7bda9645-0b5f-470a-8dd7-6af0bff4da68,exploretock,Travel,Restaurant,Check for pickup restaurant available in Bosto...,11
1,a6372f23-f462-4706-8455-5b350c46d83c,exploretock,Travel,Restaurant,Book a winery tour in Napa Valley in a winery ...,19
2,c0eeead1-f8ea-4819-a6da-ef0108b40c89,enterprise,Travel,Car rental,Sign Allan Smith for email offers with the ema...,7
3,ce34bc61-e3d2-40c8-b02b-b149efc4b115,enterprise,Travel,Car rental,Find a mini van at Brooklyn City from April 5t...,11
4,bf469f30-6628-4017-b963-672645d7feab,kohls,Shopping,Department,Find the highest rated dog collar under 10 dol...,10


In [11]:
steps.head(5)

,annotation_id,action_uid,website,step,action,op,original_op,value,target_tag,target_attributes,target_node_id,right_elements,other_elements,html_chars
0,7bda9645-0b5f-470a-8dd7-6af0bff4da68,961c3a5e-f8ce-4c71-a917-aa546dcea7fb,exploretock,1,[combobox] Reservation type -> SELECT: Pickup,SELECT,SELECT,Pickup,select,"{""backend_node_id"": ""136"", ""bounding_box_rect""...",136,1,197,38728
1,7bda9645-0b5f-470a-8dd7-6af0bff4da68,30732d9f-efee-4ba4-8b2a-a72e47d5bde6,exploretock,2,[svg] -> CLICK,CLICK,CLICK,,svg,"{""backend_node_id"": ""3113"", ""bounding_box_rect...",3113,1,195,36689
2,7bda9645-0b5f-470a-8dd7-6af0bff4da68,5bbb896b-8564-4603-9fc7-16ef2a072d56,exploretock,3,[searchbox] Find a location -> TYPE: Boston,TYPE,TYPE,Boston,input,"{""backend_node_id"": ""6068"", ""bounding_box_rect...",6068,1,208,37724
3,7bda9645-0b5f-470a-8dd7-6af0bff4da68,34eccecf-fd3e-43ca-965c-98d3be310a29,exploretock,4,[span] Boston -> CLICK,CLICK,CLICK,,button,"{""backend_node_id"": ""10413"", ""bounding_box_rec...",10413,1,219,40528
4,7bda9645-0b5f-470a-8dd7-6af0bff4da68,dc327d5f-d07b-496d-8680-400483790fce,exploretock,5,[svg] -> CLICK,CLICK,CLICK,,svg,"{""backend_node_id"": ""12210"", ""bounding_box_rec...",12210,1,197,37309


In [14]:
data = (
    steps.drop(columns=["website"])        # website is already in tasks
         .merge(tasks.rename(columns={"steps": "task_steps"}), on="annotation_id", how="left")
         [["annotation_id", "website", "domain", "subdomain", "task", "task_steps",
           "step", "action", "op", "original_op", "value",
           "target_tag", "target_attributes", "target_node_id",
           "right_elements", "other_elements", "html_chars", "action_uid"]]
         .sort_values(["domain", "website", "annotation_id", "step"])
         .reset_index(drop=True)
)

print(data.shape)
data

(774, 18)


,annotation_id,website,domain,subdomain,task,task_steps,step,action,op,original_op,value,target_tag,target_attributes,target_node_id,right_elements,other_elements,html_chars,action_uid
0,e6643cfb-567e-4e11-8cab-f85483573539,espn,Entertainment,Sports,Follow one of the team leaders of one of the N...,6,1,[link] ESPN NHL Home Page -> HOVER,CLICK,HOVER,,a,"{""backend_node_id"": ""2234"", ""bounding_box_rect...",2234,1,540,202568,caecd46f-1c1b-4494-bb4f-2d64fa469b04
1,e6643cfb-567e-4e11-8cab-f85483573539,espn,Entertainment,Sports,Follow one of the team leaders of one of the N...,6,2,[link] Teams -> CLICK,CLICK,CLICK,,a,"{""backend_node_id"": ""11887"", ""bounding_box_rec...",11887,1,645,223671,49c73a41-8cf0-4ec0-b12e-b588fa3a2320
2,e6643cfb-567e-4e11-8cab-f85483573539,espn,Entertainment,Sports,Follow one of the team leaders of one of the N...,6,3,[heading] Boston Bruins -> CLICK,CLICK,CLICK,,a,"{""backend_node_id"": ""21177"", ""bounding_box_rec...",21177,1,602,118185,3161aa29-78c0-4ab3-b776-10d894cf75fd
3,e6643cfb-567e-4e11-8cab-f85483573539,espn,Entertainment,Sports,Follow one of the team leaders of one of the N...,6,4,[link] Full Team Statistics -> CLICK,CLICK,CLICK,,a,"{""backend_node_id"": ""24577"", ""bounding_box_rec...",24577,1,548,151786,82ac4858-e319-4d30-b3e5-f4a4c395f697
4,e6643cfb-567e-4e11-8cab-f85483573539,espn,Entertainment,Sports,Follow one of the team leaders of one of the N...,6,5,[heading] David Pastrnak RW -> CLICK,CLICK,CLICK,,a,"{""backend_node_id"": ""31164"", ""bounding_box_rec...",31164,1,1607,482864,873ad00e-a6df-4834-ad94-1f8d537ee77c
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
769,f27bb47e-3aba-439a-b98a-2cf9b0e25e9c,viator,Travel,Other,Browse Tokyo's best tea related attractions in...,5,1,[span] Explore the World -> HOVER,CLICK,HOVER,,div,"{""backend_node_id"": ""303"", ""bounding_box_rect""...",303,1,473,88268,cfdab859-8ea1-4145-a761-ca9ffa100107
770,f27bb47e-3aba-439a-b98a-2cf9b0e25e9c,viator,Travel,Other,Browse Tokyo's best tea related attractions in...,5,2,[span] Tokyo -> CLICK,CLICK,CLICK,,a,"{""backend_node_id"": ""5895"", ""bounding_box_rect...",5895,1,502,94012,c9e53d13-c6fc-4af4-b8c4-45ea969dd04a
771,f27bb47e-3aba-439a-b98a-2cf9b0e25e9c,viator,Travel,Other,Browse Tokyo's best tea related attractions in...,5,3,[link] Food & Drink -> CLICK,CLICK,CLICK,,a,"{""backend_node_id"": ""9155"", ""bounding_box_rect...",9155,1,1350,309326,b1062855-4c2b-4283-9b44-d7dc68373578
772,f27bb47e-3aba-439a-b98a-2cf9b0e25e9c,viator,Travel,Other,Browse Tokyo's best tea related attractions in...,5,4,[link] Coffee & Tea -> CLICK,CLICK,CLICK,,a,"{""backend_node_id"": ""22463"", ""bounding_box_rec...",22463,1,1372,300522,46828258-db63-45df-b5d6-3807c8d23840


In [ ]:
# so 
"""
after spending 2 hours setting up the stupid lab script to create the environment and run it , and reading about mind2web

i concluded that , mind2web is for multip-step goals and generalist web ai agents , not for my use case

decision : look for another database or build a one myself
"""